# MT02 — Controllers & Cameras: a scripted pick

### Lab Description

A robot **controller** converts the high-level action you send into the low-level joint torques MuJoCo integrates. robosuite's **operational-space (BASIC) controller** lets you command the *end-effector* directly with position/orientation deltas — far easier than reasoning about seven joint torques. In this lab you use it to **script a full pick**: hover, descend, grasp, and lift the cube. You will also render the scene from several cameras.

#### Recommended Hardware
AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)
#### Software Environment
OS: Ubuntu 24.04.4 LTS \
Install [AUP learning cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=max-pro-395). After installing AUP Learning Cloud you will have the ROCm + PyTorch environment (the `auplc-mujoco-torch` course image: torch + mujoco + robosuite + gymnasium) that this notebook is built for.

## Goals
- Command the end-effector with the operational-space controller
- Render a scene from multiple named cameras
- Script a multi-phase reach–grasp–lift behaviour
- Verify the pick succeeded (cube height + task reward)

### Imports and renderer

Same setup helpers as MT01.

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

import numpy as np
import mujoco
import imageio
import robosuite as suite
from robosuite import load_composite_controller_config
from IPython.display import Video

import logging
logging.getLogger("robosuite_logs").setLevel(logging.ERROR)  # hide repetitive controller-config warnings

os.makedirs("output/videos", exist_ok=True)
print("robosuite", suite.__version__, "| mujoco", mujoco.__version__)

# robosuite's built-in camera-observation renderer can emit intermittently
# corrupted frames on this ROCm/EGL stack. We instead drive mujoco.Renderer
# directly (the reliable path) and show only the visual geom group (group 1),
# which avoids the green/blue collision shapes and looks correct.
def make_renderer(env, height=256, width=256):
    R = mujoco.Renderer(env.sim.model._model, height=height, width=width)
    opt = mujoco.MjvOption()
    opt.geomgroup[:] = 0
    opt.geomgroup[1] = 1
    return R, opt

def grab_frame(env, R, opt, camera="frontview"):
    mujoco.mj_forward(env.sim.model._model, env.sim.data._data)
    R.update_scene(env.sim.data._data, camera=camera, scene_option=opt)
    return R.render()

### Create the Lift task

In [ ]:
controller = load_composite_controller_config(controller="BASIC", robot="Panda")
env = suite.make(env_name="Lift", robots="Panda", controller_configs=controller,
                 has_renderer=False, has_offscreen_renderer=False, use_camera_obs=False,
                 control_freq=20, horizon=300)
obs = env.reset()
print("action_dim", env.action_dim)

### See the scene from several cameras

Every robosuite scene ships with named cameras. We render one frame from each so you can pick the most informative viewpoint. `frontview` shows the whole arm and table; `agentview` looks down from above; `birdview` is a wide overhead shot.

In [ ]:
import matplotlib.pyplot as plt
R, opt = make_renderer(env, 256, 256)
obs = env.reset()
cams = ["frontview", "agentview", "birdview"]
fig, axes = plt.subplots(1, len(cams), figsize=(4*len(cams), 4))
for ax, cam in zip(axes, cams):
    ax.imshow(grab_frame(env, R, opt, camera=cam)); ax.set_title(cam); ax.axis("off")
plt.tight_layout(); plt.show()

### Script a pick with end-effector control

Because the BASIC controller commands the end-effector directly, a pick is just four phases of simple position commands:
1. **Hover** a few cm above the cube (gripper open).
2. **Descend** onto the cube.
3. **Close** the gripper.
4. **Lift** straight up.

Each step we send `[dx, dy, dz, 0, 0, 0, grip]`, where the position delta is proportional to the error between the cube and the end-effector (a simple P-controller in task space).

In [ ]:
R, opt = make_renderer(env, 256, 256)
obs = env.reset()
frames = []
cube_z_start = float(obs["cube_pos"][2])

def do(action):
    global obs
    obs, reward, done, info = env.step(np.asarray(action, dtype=float))
    frames.append(grab_frame(env, R, opt, camera="frontview"))
    return reward

# Phase 1 - hover above the cube (gripper open)
for _ in range(40):
    delta = np.clip((obs["cube_pos"] + [0, 0, 0.06] - obs["robot0_eef_pos"]) * 5, -1, 1)
    do([delta[0], delta[1], delta[2], 0, 0, 0, -1.0])
# Phase 2 - descend onto the cube
for _ in range(25):
    delta = np.clip((obs["cube_pos"] - obs["robot0_eef_pos"]) * 5, -1, 1)
    do([delta[0], delta[1], delta[2], 0, 0, 0, -1.0])
# Phase 3 - close the gripper
for _ in range(15):
    do([0, 0, 0, 0, 0, 0, 1.0])
# Phase 4 - lift straight up
reward = 0.0
for _ in range(40):
    reward = do([0, 0, 0.5, 0, 0, 0, 1.0])
R.close()
print(f"cube lifted by {float(obs['cube_pos'][2]) - cube_z_start:.3f} m | final task reward {reward:.2f}")
imageio.mimsave("output/videos/mt02_scripted_pick.mp4", frames, fps=20)
print("saved", len(frames), "frames")

### Watch the scripted pick

A final reward near `1.0` and a positive lift height mean the Panda grasped and raised the cube.

In [ ]:
Video(url="output/videos/mt02_scripted_pick.mp4")

## Conclusions

Using the operational-space controller you scripted a complete reach–grasp–lift pick from a handful of end-effector commands, and rendered it cleanly from multiple cameras. Hand-scripting works but is brittle and task-specific. MT03 introduces the Gymnasium interface and reward signal that let an agent *learn* such behaviours, and MT04 trains a neural-network policy to imitate this very pick.

---

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT